In [6]:
import numpy as np
import pandas as pd
import time

from lib.envs.maze import MazeEnv

In [ ]:
class RL():
    def __init__(self, action_space, 
                 learning_rate=0.01, 
                 reward_decay =0.9, 
                 e_greedy     =0.95):
        
        self.actions = action_space
        self.lr = learning_rate
        self.gamma = reward_decay
        self.epsilon = e_greedy
        self.q_table = pd.DataFrame(columns=self.actions, dtype=np.float64)
        
    def check_state_exist(self, state):
        if state not in self.q_table.index:
            # 如果状态在当前的 Q 表中不存在,将当前状态加入 Q 表中
            new_row = pd.DataFrame(
                [[0.0] * len(self.actions)],
                columns=self.q_table.columns,
                index=[state]
            )
            self.q_table = pd.concat([self.q_table, new_row])

    def choose_action(self, observation):
        self.check_state_exist(observation)
        # 从均匀分布的[0,1]中随机采样,当小于阈值时采用选择最优行为的方式,
        # 当大于阈值时采用选择随机行为的方式,这样人为增加随机性是为了解决陷入局部最优
        
        # 最优行为概率：ε + (1-ε)/|A|
        # 其它行为概率：(1-ε)/|A|
        if np.random.rand() <= self.epsilon:
            # 选择最优行为
            state_action = self.q_table.loc[observation, :]
            # 因为一个状态下最优行为可能有多个,需要随机选择一个
            max_val = state_action.max()
            best_actions = state_action[state_action == max_val].index.tolist()
            action = np.random.choice(best_actions)
        else:
            # 选择随机行为
            action = np.random.choice(self.actions)
        return action

    def learn(self, *args):
        pass

In [8]:
# 离线策略 Q-Learning
class QLearningTable(RL):  # 继承 RL 基类
    def __init__(self, actions, 
                 learning_rate=0.01, 
                 reward_decay =0.9, 
                 e_greedy     =0.9):
        super().__init__(actions, learning_rate, reward_decay, e_greedy)

    def learn(self, s, a, r, s_):
        self.check_state_exist(s)
        self.check_state_exist(s_)
        
        q_predict = self.q_table.loc[s, a]

        if s_ != 'terminal':
            # 使用公式: Q_target = r + γ maxQ(s',a')
            q_target = r + self.gamma * self.q_table.loc[s_, :].max()
        else:
            q_target = r

        # 更新公式: Q(s,a) ← Q(s,a) + α(r + γ max Q(s',a') − Q(s,a))
        self.q_table.loc[s, a] += self.lr * (q_target - q_predict)

    def get_action(self, q_table, state):
        # 选择最优行为
        state_action = q_table.loc[state, :]
        max_val = state_action.max()
        
        idxs = [i for i, v in enumerate(state_action) if v == max_val]
        idxs.sort()
        return tuple(idxs)
    
    def get_policy(self, q_table, env, rows=5, cols=5, pixels=40, origin=20):
        policy = []
        # 预计算所有陷阱和宝藏坐标
        hell_positions = [
            env.canvas.coords(env.hell1), 
            env.canvas.coords(env.hell2),
            env.canvas.coords(env.hell3), 
            env.canvas.coords(env.hell4),
            env.canvas.coords(env.hell5), 
            env.canvas.coords(env.hell6),
            env.canvas.coords(env.hell7), 
            env.canvas.coords(env.oval)
        ]
        
        for i in range(rows):
            for j in range(cols):
                # 求出每个格子的状态
                item_center_x = (j * pixels + origin)
                item_center_y = (i * pixels + origin)
                item_state = [item_center_x - 15.0, 
                              item_center_y - 15.0, 
                              item_center_x + 15.0, 
                              item_center_y + 15.0]
                                
                if item_state in hell_positions:
                    policy.append(-1)
                    continue
                
                if str(item_state) not in q_table.index:
                    policy.append((0, 1, 2, 3))
                    continue
                # 选择最优行为
                item_action_max = self.get_action(q_table, str(item_state))
                policy.append(item_action_max)
        return policy

In [9]:
def update(agent, env, episodes=100):
    for episode in range(episodes):
        # 初始化状态
        observation = env.reset()
        c = 0
        tmp_policy = {}
        
        while True:
            # 渲染当前环境
            env.render()

            # 基于当前状态选择行为
            action = agent.choose_action(str(observation))

            state_item = tuple(observation)
            tmp_policy[state_item] = action

            # 采取行为获得下一个状态和回报及是否终止
            next_obs, reward, done, oval_flag = env.step(action)
            # 根据当前的变化开始更新 Q
            agent.learn(str(observation), action, reward, str(next_obs))

            # 改变状态和行为
            observation = next_obs
            c += 1
            # 如果为终止状态,则结束当前的局数
            if done:
                break
        print(f'第 {episode + 1} 局结束')

    # 开始输出最终的 Q 表
    q_table_result = agent.q_table
    print(q_table_result)

    # 使用 Q 表输出各状态的最优策略
    policy = agent.get_policy(q_table_result, env)
    policy_result = np.array(policy, dtype=object).reshape(5,5)
    print('\n最优策略 (0:上 1:下 2:左 3:右, -1:陷阱/宝藏):')
    print(policy_result)

    env.render_by_policy(policy_result)
    # env.destroy()

In [10]:
env = MazeEnv()
agent = QLearningTable(actions=list(range(env.n_actions))) 
env.after(100, lambda: update(agent, env))
env.mainloop()

第 1 局结束
第 2 局结束
第 3 局结束
第 4 局结束
第 5 局结束
第 6 局结束
第 7 局结束
第 8 局结束
第 9 局结束
第 10 局结束
第 11 局结束
第 12 局结束
第 13 局结束
第 14 局结束
第 15 局结束
第 16 局结束
第 17 局结束
第 18 局结束
第 19 局结束
第 20 局结束
第 21 局结束
第 22 局结束
第 23 局结束
第 24 局结束
第 25 局结束
第 26 局结束
第 27 局结束
第 28 局结束
第 29 局结束
第 30 局结束
第 31 局结束
第 32 局结束
第 33 局结束
第 34 局结束
第 35 局结束
第 36 局结束
第 37 局结束
第 38 局结束
第 39 局结束
第 40 局结束
第 41 局结束
第 42 局结束
第 43 局结束
第 44 局结束
第 45 局结束
第 46 局结束
第 47 局结束
第 48 局结束
第 49 局结束
第 50 局结束
第 51 局结束
第 52 局结束
第 53 局结束
第 54 局结束
第 55 局结束
第 56 局结束
第 57 局结束
第 58 局结束
第 59 局结束
第 60 局结束
第 61 局结束
第 62 局结束
第 63 局结束
第 64 局结束
第 65 局结束
第 66 局结束
第 67 局结束
第 68 局结束
第 69 局结束
第 70 局结束
第 71 局结束
第 72 局结束
第 73 局结束
第 74 局结束
第 75 局结束
第 76 局结束
第 77 局结束
第 78 局结束
第 79 局结束
第 80 局结束
第 81 局结束
第 82 局结束
第 83 局结束
第 84 局结束
第 85 局结束
第 86 局结束
第 87 局结束
第 88 局结束
第 89 局结束
第 90 局结束
第 91 局结束
第 92 局结束
第 93 局结束
第 94 局结束
第 95 局结束
第 96 局结束
第 97 局结束
第 98 局结束
第 99 局结束
第 100 局结束
                                     0         1         2         3
[5.0, 5.0, 35.0, 35.0]       -0.915548 